In [1]:
import json
import torch
import torch.nn as nn
import math
from huggingface_hub import login, hf_hub_download
from datasets import Dataset
from transformers import PreTrainedTokenizerFast, TrainingArguments, Trainer, DataCollatorForTokenClassification
from safetensors.torch import load_file


In [ ]:
login(token=os.environ["HF_TOKEN"])

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MODEL_REPO = "A-240242/legal-gqa-checkpoints" 
HF_TOKENIZER_REPO= "A-240242/legal-modern-decoder-checkpoints"

In [3]:
def load_phase2_file(filename):
    path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=f"Phase_2_LNER/Train/{filename}", repo_type="dataset", token=HF_TOKEN)
    if filename.endswith(".txt"):
        with open(path, "r", encoding="utf-8") as f:
            return [line.strip() for line in f.readlines() if line.strip()]
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("Downloading Phase 2 Data...")
# Load and merge text dictionaries
text_data = load_phase2_file("fold1-text.json")
text_data.update(load_phase2_file("fold2-text.json"))

# Load and merge gold annotation dictionaries
gold_data = load_phase2_file("fold1-gold.json")
gold_data.update(load_phase2_file("fold2-gold.json"))

entity_types = load_phase2_file("labels.txt")
print(f"Loaded {len(text_data)} total documents across both folds.")

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

Loaded 70 total documents across both folds.


In [4]:
# 1. Build BIO Schema
LABEL_LIST = ["O"]
for entity in entity_types:
    LABEL_LIST.extend([f"B-{entity}", f"I-{entity}"])

LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
NUM_LABELS = len(LABEL_LIST)

# 2. Load Phase 1 Tokenizer
tokenizer = PreTrainedTokenizerFast.from_pretrained(HF_TOKENIZER_REPO, token=HF_TOKEN)
tokenizer.padding_side = "right"

# 3. Manual Sliding Window Alignment
def align_spans_to_tokens(raw_text, gold_spans, tokenizer, max_length=1024, stride=256):
    # Tokenize the ENTIRE document without truncation
    tokenized_full = tokenizer(raw_text, return_offsets_mapping=True, truncation=False)
    full_ids = tokenized_full["input_ids"]
    full_offsets = tokenized_full["offset_mapping"]
    
    chunked_data = {"input_ids": [], "attention_mask": [], "labels": []}
    step = max_length - stride
    
    # Manually slide a window across the massive token arrays
    for i in range(0, len(full_ids), step):
        chunk_ids = full_ids[i : i + max_length]
        chunk_offs = full_offsets[i : i + max_length]
        
        pad_len = max_length - len(chunk_ids)
        chunk_ids = chunk_ids + [tokenizer.pad_token_id] * pad_len
        chunk_offs = chunk_offs + [(0, 0)] * pad_len
        chunk_mask = [1] * (max_length - pad_len) + [0] * pad_len
        
        chunk_labels = []
        current_entity_id = None
        
        for tok_start, tok_end in chunk_offs:
            if tok_start == tok_end:
                chunk_labels.append(-100)
                continue

            sub_text = raw_text[tok_start:tok_end]
            clean_tok_start = tok_start + (len(sub_text) - len(sub_text.lstrip()))
            clean_tok_end = tok_end - (len(sub_text) - len(sub_text.rstrip()))

            matched_label, matched_span_idx = "O", None
            
            for s_idx, span in enumerate(gold_spans):
                g_start, g_end, g_label = span[:3] 
                if not (clean_tok_end <= g_start or clean_tok_start >= g_end):
                    matched_label = g_label
                    matched_span_idx = s_idx
                    break

            if matched_label == "O":
                chunk_labels.append(LABEL2ID["O"])
                current_entity_id = None
            else:
                if current_entity_id == matched_span_idx:
                    chunk_labels.append(LABEL2ID[f"I-{matched_label}"])
                else:
                    chunk_labels.append(LABEL2ID[f"B-{matched_label}"])
                    current_entity_id = matched_span_idx
                    
        chunked_data["input_ids"].append(chunk_ids)
        chunked_data["attention_mask"].append(chunk_mask)
        chunked_data["labels"].append(chunk_labels)

    return chunked_data

tokenizer_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

In [5]:
print("Processing and Aligning Training Spans...")
all_features = {"input_ids": [], "attention_mask": [], "labels": []}

for doc_id, raw_text in text_data.items():
    gold_spans = gold_data.get(doc_id, [])
    aligned = align_spans_to_tokens(raw_text, gold_spans, tokenizer)
    
    all_features["input_ids"].extend(aligned["input_ids"])
    all_features["attention_mask"].extend(aligned["attention_mask"])
    all_features["labels"].extend(aligned["labels"])

train_dataset = Dataset.from_dict(all_features)
print(f"Successfully chunked into {len(train_dataset)} training sequences.")

Processing and Aligning Training Spans...
Successfully chunked into 636 training sequences.


In [6]:
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import PretrainedConfig, Trainer, TrainingArguments, DataCollatorForTokenClassification
from transformers.modeling_outputs import TokenClassifierOutput
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download

# ==========================================
# 1. BASE CONFIGURATION & LORMA ADAPTER
# ==========================================
class ModernDecoderConfig(PretrainedConfig):
    model_type = "modern_decoder"
    def __init__(
        self,
        vocab_size=32000,
        hidden_size=768,
        intermediate_size=2048,
        num_hidden_layers=12,
        num_attention_heads=12,
        rms_norm_eps=1e-5,
        max_position_embeddings=2048,
        rope_theta=10000.0,
        pad_token_id=0,
        eos_token_id=0,
        **kwargs,
    ):
        super().__init__(**kwargs, pad_token_id=pad_token_id, eos_token_id=eos_token_id)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.rms_norm_eps = rms_norm_eps
        self.max_position_embeddings = max_position_embeddings
        self.rope_theta = rope_theta

class LormaAdapter(nn.Module):
    def __init__(self, hidden_size, rank=128, alpha=128):
        super().__init__()
        self.scaling = alpha / rank
        self.A = nn.Linear(hidden_size, rank, bias=False)
        self.B = nn.Linear(rank, hidden_size, bias=False)
        nn.init.normal_(self.A.weight, std=1e-3)
        nn.init.zeros_(self.B.weight)
        
    def forward(self, x):
        return self.scaling * self.B(self.A(x))

# ==========================================
# 2. MODERN DECODER COMPONENTS WITH RoPE
# ==========================================
class RMSNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(hidden_size))

    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_position_embeddings=2048, base=10000.0):
        super().__init__()
        self.dim = dim
        self.max_position_embeddings = max_position_embeddings
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        
        t = torch.arange(max_position_embeddings, dtype=torch.float32)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, x, seq_len=None):
        if seq_len > self.max_position_embeddings:
            self.max_position_embeddings = seq_len
            t = torch.arange(self.max_position_embeddings, device=x.device, dtype=torch.float32)
            freqs = torch.outer(t, self.inv_freq.to(x.device))
            emb = torch.cat((freqs, freqs), dim=-1)
            self.cos_cached = emb.cos().to(x.dtype)
            self.sin_cached = emb.sin().to(x.dtype)
            
        return (
            self.cos_cached[:seq_len, :].to(x.device),
            self.sin_cached[:seq_len, :].to(x.device)
        )

def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(1)
    sin = sin.unsqueeze(0).unsqueeze(1)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

class SwiGLUMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class ModernAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = self.hidden_size // self.num_heads
        
        self.q_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.k_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.v_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.o_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.rotary_emb = RotaryEmbedding(self.head_dim, config.max_position_embeddings, config.rope_theta)

    def forward(self, hidden_states, attention_mask=None):
        bsz, q_len, _ = hidden_states.size()
        q = self.q_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        cos, sin = self.rotary_emb(v, seq_len=q_len)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        
        # CRITICAL FIX FOR NER: Set is_causal=False to grant bidirectional spatial awareness
        attn_output = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        return self.o_proj(attn_output.transpose(1, 2).contiguous().view(bsz, q_len, self.hidden_size))

# ==========================================
# 3. LORMA INJECTION LAYER
# ==========================================
class ModernLormaDecoderLayer(nn.Module):
    def __init__(self, config, rank=128):
        super().__init__()
        self.self_attn = ModernAttention(config)
        self.lorma_attn = LormaAdapter(config.hidden_size, rank=rank)
        
        self.mlp = SwiGLUMLP(config)
        self.lorma_mlp = LormaAdapter(config.hidden_size, rank=rank)
        
        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(self, hidden_states, attention_mask=None):
        # Sequential post-projection adapter scaling
        attn_out = self.self_attn(self.input_layernorm(hidden_states), attention_mask)
        hidden_states = hidden_states + attn_out + self.lorma_attn(attn_out)
        
        mlp_out = self.mlp(self.post_attention_layernorm(hidden_states))
        hidden_states = hidden_states + mlp_out + self.lorma_mlp(mlp_out)
        return hidden_states

# ==========================================
# 4. FULL MODERN DECODER NER MODEL
# ==========================================
class ModernDecoderLormaForNER(nn.Module):
    def __init__(self, config, rank=128):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size, padding_idx=config.pad_token_id)
        self.layers = nn.ModuleList([ModernLormaDecoderLayer(config, rank=rank) for _ in range(config.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.classifier = nn.Linear(config.hidden_size, NUM_LABELS, bias=True)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        hidden_states = self.embed_tokens(input_ids)
        
        for layer in self.layers:
            hidden_states = layer(hidden_states, attention_mask)
            
        logits = self.classifier(self.norm(hidden_states))
        
        loss = None
        if labels is not None:
            # Maintain 3.0x multiplier on non-O entity classes to balance signal
            class_weights = torch.ones(logits.size(-1), device=logits.device) * 3.0
            class_weights[0] = 1.0  
            loss_fn = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)
            loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
            
        return TokenClassifierOutput(loss=loss, logits=logits)

# ==========================================
# 5. INITIALIZATION & WEIGHT MAPPING
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Initializing Bidirectional Modern Decoder with LORMA...")
config = ModernDecoderConfig(
    vocab_size=len(tokenizer), 
    pad_token_id=tokenizer.pad_token_id, 
    eos_token_id=tokenizer.eos_token_id
)
model = ModernDecoderLormaForNER(config, rank=128)

print("Fetching Phase 1 Pretrained Checkpoint...")
HF_MODEL_REPO = "A-240242/legal-modern-decoder-checkpoints"
model_weight_path = hf_hub_download(repo_id=HF_MODEL_REPO, filename="model.safetensors", token=HF_TOKEN)
model.load_state_dict(load_file(model_weight_path), strict=False)

# 1. Freeze all base parameters from Phase 1
for param in model.parameters(): 
    param.requires_grad = False

# 2. Unfreeze the newly injected LORMA matrices and NER classifier head
for layer in model.layers:
    for param in layer.lorma_attn.parameters(): param.requires_grad = True
    for param in layer.lorma_mlp.parameters(): param.requires_grad = True

for param in model.classifier.parameters(): 
    param.requires_grad = True

model.to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params / 1e6:.2f}M")
print(f"Trainable LORMA Parameters: {trainable_params:,} ({(trainable_params/total_params)*100:.2f}%)")

Initializing Bidirectional Modern Decoder with LORMA...
Fetching Phase 1 Pretrained Checkpoint...


model.safetensors: reconstructing file:   0%|          |  0.00B /  536MB            

model.safetensors: downloading bytes:           |  0.00B            

Total Parameters: 114.27M
Trainable LORMA Parameters: 4,737,817 (4.15%)


In [7]:
# ==========================================
# 6. TRAINING EXECUTION
# ==========================================
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./phase2_modern_decoder_lorma",
    report_to="none",
    num_train_epochs=25, 
    per_device_train_batch_size=2,         # Safe limit to prevent OOM
    gradient_accumulation_steps=8,         # Effective batch size = 16
    learning_rate=1e-3,                    # Aggressive LR for zero-initialized LORMA blocks
    weight_decay=0.01,
    logging_steps=100,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("\nCommencing Bidirectional Modern Decoder LORMA Training...")
start_time = time.time()
trainer.train()
print(f"Finished in {(time.time() - start_time) / 60:.2f} minutes")


Commencing Bidirectional Modern Decoder LORMA Training...


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
100,5.203737
200,0.966559
300,0.343968
400,0.082708
500,0.010639


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

Finished in 36.06 minutes


In [12]:
# 4. Inference Decoding Function with Relaxed Span Merger
def decode_token_predictions_to_spans(raw_text, chunks_preds, chunks_offsets):
    extracted_spans = []

    for preds, offsets in zip(chunks_preds, chunks_offsets):
        current_entity = None 

        for pred_id, (tok_start, tok_end) in zip(preds, offsets):
            if tok_start == tok_end:
                continue

            tag = ID2LABEL.get(pred_id, "O")

            if tag.startswith("B-"):
                if current_entity is not None:
                    extracted_spans.append(current_entity)
                current_entity = {"label": tag[2:], "start": tok_start, "end": tok_end}

            elif tag.startswith("I-"):
                label_type = tag[2:]
                if current_entity is not None and current_entity["label"] == label_type:
                    current_entity["end"] = tok_end
                else:
                    if current_entity is not None:
                        extracted_spans.append(current_entity)
                    current_entity = {"label": label_type, "start": tok_start, "end": tok_end}
            else: 
                if current_entity is not None:
                    extracted_spans.append(current_entity)
                    current_entity = None

        if current_entity is not None:
            extracted_spans.append(current_entity)

   # --- THE FIX: Smart Gap Merger ---
    merged_spans = []
    for span in extracted_spans:
        if not merged_spans:
            merged_spans.append(span)
            continue
        
        prev = merged_spans[-1]
        gap = span["start"] - prev["end"]
        
        if span["label"] == prev["label"] and 0 <= gap <= 12:
            # Extract the exact text between the two fragments
            gap_text = raw_text[prev["end"]:span["start"]].lower().strip()
            
            # 1. Is the gap purely whitespace, punctuation, OR numbers (to catch missed months like ".10.")?
            is_safe_gap = all(c in ".,/- &0123456789" for c in gap_text)
            
            # 2. Is the gap a common legal connector?
            is_connector = gap_text in ["", "of", "v", "v.", "vs", "vs.", "and", "&"]
            
            if is_safe_gap or is_connector:
                prev["end"] = span["end"] 
                continue # Safely stitched!
            
        merged_spans.append(span)

    # Clean, Snap Boundaries, and Deduplicate
    final_spans = []
    seen = set()

    for span in merged_spans:
        s, e, label = span["start"], span["end"], span["label"]
        
        clean_start = s
        clean_end = e
        
        while clean_start < clean_end and (raw_text[clean_start].isspace() or raw_text[clean_start] in ".,;:-"):
            clean_start += 1
        while clean_end > clean_start and (raw_text[clean_end-1].isspace() or raw_text[clean_end-1] in ".,;:-"):
            clean_end -= 1

        if clean_start >= clean_end:
            continue

        span_tuple = (clean_start, clean_end, label)
        if span_tuple not in seen:
            seen.add(span_tuple)
            final_spans.append([clean_start, clean_end, label])

    final_spans.sort(key=lambda x: (x[0], x[1]))
    return final_spans

In [9]:
# # --- Verification Test ---
# test_doc = "IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 1128 OF 2026 RAMESH ... APPELLANT"
# test_gold = [
#     [7, 29, "COURT"],      # "SUPREME COURT OF INDIA"
#     [62, 98, "CASENO"],     # "CRIMINAL APPEAL NO. 1128 OF 2026"
#     [99, 105, "APP"]        # "RAMESH"
# ]

# # 1. Encode
# features = align_spans_to_tokens(test_doc, test_gold, tokenizer, max_length=128, stride=32)

# # 2. Simulate perfect model predictions by re-using gold label targets
# simulated_preds = [
#     [lbl if lbl != -100 else 0 for lbl in chunk_lbls] 
#     for chunk_lbls in features["labels"]
# ]

# # 3. Decode
# recovered_spans = decode_token_predictions_to_spans(
#     test_doc, 
#     features["input_ids"], 
#     features["offset_mapping"], 
#     simulated_preds
# )

# # 4. Assert Exact Character-Level Match
# print("Original Gold Spans: ", test_gold)
# print("Recovered Spans:     ", recovered_spans)

# assert recovered_spans == test_gold, "Alignment failed! Offsets do not match exactly."
# print("\nAlignment pipeline verified: 100% character-offset correctness achieved.")

In [13]:
# --- Inference & CodaBench Submission ---
import zipfile
import json
import torch
import gc
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
import os

# 1. CRITICAL: Flip padding back to right-side for NER alignment
tokenizer.padding_side = "right"

print("Loading Test Data...")
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO, 
    filename="Phase_2_LNER/Test/fold3-text.json", 
    repo_type="dataset", 
    token=HF_TOKEN
)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Successfully loaded {len(test_data)} documents from the test fold.")

# Rebuild ID2LABEL locally
LABEL_LIST = ["O"]
for entity in entity_types:
    LABEL_LIST.extend([f"B-{entity}", f"I-{entity}"])
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}

model.eval()
model.to("cuda")
predictions = {}

chunk_size = 1024
stride = 256
step = chunk_size - stride

for doc_id, raw_text in tqdm(test_data.items(), desc="Generating Modern Decoder Spans"):
    torch.cuda.empty_cache()
    gc.collect()
    
    try:
        # Tokenize document without truncation
        tokenized = tokenizer(raw_text, return_offsets_mapping=True, truncation=False)
        full_ids = tokenized["input_ids"]
        full_offsets = tokenized["offset_mapping"]
        
        doc_preds = []
        doc_offsets = []
        
        # Slide padded window across document
        for i in range(0, len(full_ids), step):
            raw_chunk_ids = full_ids[i : i + chunk_size]
            raw_chunk_offs = full_offsets[i : i + chunk_size]
            actual_len = len(raw_chunk_ids)
            
            # Pad to 1024 matching the training configuration
            pad_len = chunk_size - actual_len
            padded_ids = raw_chunk_ids + [tokenizer.pad_token_id] * pad_len
            mask = [1] * actual_len + [0] * pad_len
            
            input_tensor = torch.tensor([padded_ids], dtype=torch.long, device="cuda")
            attn_mask_tensor = torch.tensor([mask], dtype=torch.long, device="cuda")
            
            # Model Prediction
            with torch.inference_mode():
                outputs = model(input_ids=input_tensor, attention_mask=attn_mask_tensor)
                logits = outputs.logits
                preds = torch.argmax(logits, dim=-1).cpu().numpy()[0]
                
            # Slice off the padding before decoding
            doc_preds.append(preds[:actual_len])
            doc_offsets.append(raw_chunk_offs)
            
        # Decode tokens back to character spans
        extracted_spans = decode_token_predictions_to_spans(raw_text, doc_preds, doc_offsets)
        predictions[doc_id] = extracted_spans

    except Exception as e:
        print(f"\nError processing document {doc_id}: {str(e)}")
        predictions[doc_id] = []

# 2. Save to Dedicated Output Directory
output_dir = "./phase2_modern_decoder_lorma"
os.makedirs(output_dir, exist_ok=True)
PREDICTION_PATH = os.path.join(output_dir, "prediction.json")
SUBMISSION_ZIP = os.path.join(output_dir, "submission.zip")

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print(f"\nFinal sanity check: Saved {len(predictions.keys())} document IDs to prediction.json")
print(f"L-NER Predictions complete! Submission saved to: {SUBMISSION_ZIP}")

Loading Test Data...
Successfully loaded 35 documents from the test fold.


Generating Modern Decoder Spans:   0%|          | 0/35 [00:00<?, ?it/s]


Final sanity check: Saved 35 document IDs to prediction.json
L-NER Predictions complete! Submission saved to: ./phase2_modern_decoder_lorma/submission.zip


In [14]:
# Run this in a new cell to diagnose boundary artifacts
first_doc_id = list(predictions.keys())[0]
raw_doc_text = test_data[first_doc_id]

print(f"--- Diagnostic: Top 20 Predictions for Doc {first_doc_id} ---")
for span in predictions[first_doc_id][:20]:
    start, end, label = span
    extracted_text = raw_doc_text[start:end]
    # Using repr() to expose hidden trailing spaces or newlines (e.g. 'Court \n')
    print(f"[{label: <10}] : {repr(extracted_text)}")

--- Diagnostic: Top 20 Predictions for Doc 1128835 ---
[COURT     ] : 'COURT OF INDIA'
[CASENO    ] : 'CIVIL APPEAL NO.6'
[CASENO    ] : '2008'
[APP       ] : 'FABS PVT. LTD'
[RESP      ] : 'STATE OF HARYANA'
[CASENO    ] : 'CIVIL APPEAL NO.6581 OF 2008'
[CASENO    ] : '15544'
[APP       ] : 'PAT TEACHERS (RECOGNISED'
[APP       ] : 'LTD'
[RESP      ] : 'STATE OF HARYANA'
[CASENO    ] : '30 OF 2007'
[CASENO    ] : 'CIVIL APPEAL NO.6580 OF 2008'
[CASENO    ] : 'NO. 15449 OF 2004'
[APP       ] : 'CO FABS PVT. LTD'
[RESP      ] : 'STATE OF HARYANA'
[JUDGE     ] : 'K'
[JUDGE     ] : 'HAKKER'
[DATE      ] : 'April 02, 2004'
[COURT     ] : 'High Court of Punjab & Haryana'
[CASENO    ] : 'Civil Writ Petition Nos. 1853 of 2003'
